In [95]:
#Membuat spark session dan import berbagai library
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, avg, count, col
import os
import shutil
import time
import pandas as pd

spark =SparkSession.builder.appName("TugasMandiriPertemuan10").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.shuffle.partitions","4")

print("Sparksession siap. Versi spark ",spark.version)

Sparksession siap. Versi spark  3.5.9


In [96]:
##A. Menyiapkan Streaming Source
#Menyiapkan folder kosong sebagai "sumber data streaming"
folder_stream = "tugas10_stream"
folder_checkpoint="checkpoint_tugas10"

shutil.rmtree(folder_stream, ignore_errors=True) #menghapus folder stream sisa
shutil.rmtree(folder_checkpoint, ignore_errors=True) #menghapus folder checkpoint sisa

os.makedirs(folder_stream, exist_ok=True) #membuat folder stream di lokal

#Mendifinisikan skema eksplisit (StructType)
skema = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])
print(f"Folder streaming '{folder_stream}' siap'")

#Mendefinisikan readStream
df_stream = spark.readStream\
    .format("csv")\
    .schema(skema)\
    .option("header", True)\
    .load(folder_stream)
print("Streming dataframe berhasil didefinisikan")
print("df_stream.isStreaming: ",df_stream.isStreaming)

Folder streaming 'tugas10_stream' siap'
Streming dataframe berhasil didefinisikan
df_stream.isStreaming:  True


In [97]:
##B. Agregasi per Cabang
#Menambahkan kolom turunan total_harga

df_stream = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

#Membuat agregasi groupBy 'nama_cabang' menghitung total_harga dan jumlah pesanan
agregasi_stream = df_stream.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)
print("Transformasi berhasil dibuat")
#Mendefinisikan output sink
query = agregasi_stream.writeStream\
    .outputMode("complete")\
    .format("memory")\
    .queryName("agregasi_cabang")\
    .start()

print("Streaming query dimulai. Status : ",query.status)

Transformasi berhasil dibuat
Streaming query dimulai. Status :  {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [98]:
##C. Agregasi Berbasis Window Waktu

#Mengelompokkan  berdasarkan window waktu 2 menit, dihitung dari kolom waktu_pesan
agregasi_window = df_stream.groupBy(
    window(col("waktu_pesan"), "2 minute"),
    
).agg(
    spark_sum("total_harga").alias("total_harga"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query2=agregasi_window.writeStream\
    .outputMode("complete")\
    .format("memory")\
    .queryName("agregasi_waktu")\
    .start()
print("Streaming query window waktu dimulai")
print("Streaming query dimulai. Status : ",query2.status)

Streaming query window waktu dimulai
Streaming query dimulai. Status :  {'message': 'Getting offsets from FileStreamSource[file:/home/celloz/praktikum-bigdata/tugas10_stream]', 'isDataAvailable': False, 'isTriggerActive': True}


In [99]:
##D. Menguji dengan data simulasi
#Batch 1
pd.DataFrame({
    "pesanan_id":["TX1","TX2","TX3"],
    "nama_cabang":["Magelang"," Semarang","Solo"],
    "nama_menu":["oseng-oseng iga", "nasi uduk", "nasi liwet"],
    "qty":[2,5,3],
    "harga_satuan":[32000, 17000,15000],
    "waktu_pesan": ["2026-11-06 14:30:00", "2026-11-06 14:31:15", "2026-11-06 14:32:00"]
        }).to_csv(f"{folder_stream}/batch_1.csv", index= False)

print("Batch 1 dikirim, spark loading....")
time.sleep(7)

spark.sql("select * from agregasi_cabang").show()

Batch 1 dikirim, spark loading....
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Semarang|      85000|             1|
|       Solo|      45000|             1|
|   Magelang|      64000|             1|
+-----------+-----------+--------------+



In [100]:
#Batch 2
pd.DataFrame({
    "pesanan_id":["TX4","TX5","TX6","TX7"],
    "nama_cabang":["Semarang"," Surabaya","Solo","Magelang"],
    "nama_menu":["seblak Hot", "cakwe", "nasi liwet","nasi godog"],
    "qty":[1,3,2,1],
    "harga_satuan":[23000, 12000,15000, 22000],
    "waktu_pesan": ["2026-11-06 14:35:00","2026-11-06 14:36:30","2026-11-06 14:38:15","2026-11-06 14:40:00"]
        }).to_csv(f"{folder_stream}/batch_2.csv", index= False)

print("Batch 2 dikirim, spark loading....")
time.sleep(7)

spark.sql("select * from agregasi_cabang").show()
#spark.sql("select window.start, window.end, nama_cabang, total_harga, jumlah_pesanan from agregasi_waktu order by window.start").show(truncate=False)
    

Batch 2 dikirim, spark loading....
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Semarang|      23000|             1|
|   Surabaya|      36000|             1|
|   Semarang|      85000|             1|
|       Solo|      75000|             2|
|   Magelang|      86000|             2|
+-----------+-----------+--------------+



In [102]:
#Mengalirkan batch ke 3
pd.DataFrame({
    "pesanan_id":["TX8","TX9","TX10"],
    "nama_cabang":["Semarang"," Surabaya","Magelang"],
    "nama_menu":["seblak hot", "cheese cake","nasi goreng"],
    "qty":[1,2,1],
    "harga_satuan":[23000, 20000,18000],
    "waktu_pesan": ["2026-11-06 14:37:00","2026-11-06 14:43:30","2026-11-06 14:45:15"]
        }).to_csv(f"{folder_stream}/batch_3.csv", index= False)

print("Batch 3 dikirim, spark loading....")
time.sleep(7)

spark.sql("select * from agregasi_cabang").show()
spark.sql("select window.start, window.end, total_harga, jumlah_pesanan from agregasi_waktu order by window.start").show(truncate=False)
    

Batch 3 dikirim, spark loading....
+-----------+-----------+--------------+
|nama_cabang|total_harga|jumlah_pesanan|
+-----------+-----------+--------------+
|   Semarang|      46000|             2|
|   Surabaya|      76000|             2|
|   Semarang|      85000|             1|
|       Solo|      75000|             2|
|   Magelang|     104000|             3|
+-----------+-----------+--------------+

+-------------------+-------------------+-----------+--------------+
|start              |end                |total_harga|jumlah_pesanan|
+-------------------+-------------------+-----------+--------------+
|2026-11-06 14:30:00|2026-11-06 14:32:00|149000     |2             |
|2026-11-06 14:32:00|2026-11-06 14:34:00|45000      |1             |
|2026-11-06 14:34:00|2026-11-06 14:36:00|23000      |1             |
|2026-11-06 14:36:00|2026-11-06 14:38:00|59000      |2             |
|2026-11-06 14:38:00|2026-11-06 14:40:00|30000      |1             |
|2026-11-06 14:40:00|2026-11-06 14:42:00|22

In [103]:
## E. Sink Berbasis Berkas (Challange)
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Streaming query ke parquet siap. Status : ",query3.status)

Streaming query ke parquet siap. Status :  {'message': 'Initializing sources', 'isDataAvailable': False, 'isTriggerActive': False}


In [104]:
# Mengirimkan batch ke 4
pd.DataFrame({
    "pesanan_id": ["TX8", "TX9", "TX10"],
    "nama_cabang": ["Magelang", "Solo", "Semarang"],
    "nama_menu": ["es teh", "nasi goreng", "kopi hitam"],
    "qty": [2, 1, 3],
    "harga_satuan": [5000, 18000, 10000],
    "waktu_pesan": ["2026-11-06 14:42:00", "2026-11-06 14:43:30", "2026-11-06 14:45:15"]
}).to_csv(f"{folder_stream}/batch_4.csv", index=False)

print("Batch 4 dikirim ke stream, menunggu Spark menulis ke Parquet...")
time.sleep(7)

Batch 4 dikirim ke stream, menunggu Spark menulis ke Parquet...


In [105]:
#Verifikasi
!ls -l tugas10_output

df_parquet=spark.read.parquet("tugas10_output")
df_parquet.show()

drwxr-xr-x    - celloz  6 Nov 17:27  _spark_metadata
.rw-r--r--   24 celloz  6 Nov 17:27  .part-00000-1ce6aedd-fadd-4638-9dab-0750b39a89b1-c000.snappy.parquet.crc
.rw-r--r--   28 celloz  6 Nov 17:27  .part-00000-c052e74d-a3d1-4bdd-9a92-aca2ad2373ca-c000.snappy.parquet.crc
.rw-r--r--   28 celloz  6 Nov 17:27  .part-00001-13b1acc9-3e4e-44a5-87b9-2d0e93d3bfd7-c000.snappy.parquet.crc
.rw-r--r--   28 celloz  6 Nov 17:27  .part-00002-fc55e56a-1d21-4a66-a16b-430d4cba5713-c000.snappy.parquet.crc
.rw-r--r-- 2,0k celloz  6 Nov 17:27  part-00000-1ce6aedd-fadd-4638-9dab-0750b39a89b1-c000.snappy.parquet
.rw-r--r-- 2,1k celloz  6 Nov 17:27  part-00000-c052e74d-a3d1-4bdd-9a92-aca2ad2373ca-c000.snappy.parquet
.rw-r--r-- 2,1k celloz  6 Nov 17:27  part-00001-13b1acc9-3e4e-44a5-87b9-2d0e93d3bfd7-c000.snappy.parquet
.rw-r--r-- 2,1k celloz  6 Nov 17:27  part-00002-fc55e56a-1d21-4a66-a16b-430d4cba5713-c000.snappy.parquet
+----------+-----------+---------------+---+------------+-------------------+-

In [108]:
query.stop()
print("kueri dihentikan", query.status)
query2.stop()
print("kueri 2 dihentikan",query2.status)
query3.stop()
print("kueri 3 dihentikan",query3.status)


kueri dihentikan {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
kueri 2 dihentikan {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
kueri 3 dihentikan {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [107]:
##Menghentikan spark session
spark.stop
print("spark sudah bekerja dengan baik")

spark sudah bekerja dengan baik
